# Political Response Clarity with DistilBERT

Fine-tuning an efficient transformer on question-answer pairs from QEvasion, with class-balanced loss, stratified validation, early stopping, and error analysis.

## Results status

This cleaned source does not claim metrics because the Kaggle export did not preserve a complete, trustworthy result set. Rerun all cells before publishing scores or plots.

In [ ]:
import os, random, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
print("All imports successful ✓")

In [ ]:
# ============================================================
# Configuration
# ============================================================
MODEL_NAME       = "distilbert/distilbert-base-uncased"
MAX_LEN          = 256
BATCH_SIZE       = 16
LEARNING_RATE    = 3e-5
EPOCHS           = 5
WARMUP_RATIO     = 0.1
WEIGHT_DECAY     = 0.01
SEED             = 42
NUM_CLASSES      = 3
PATIENCE         = 2
GRAD_CLIP        = 1.0
USE_TOKEN_TYPE_IDS = False

LABEL2ID = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

SUBMISSION_FILE = "submission_distilbert-base-uncased.csv"

In [ ]:
def set_seed(seed):
    '''Set random seed for full reproducibility.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1 · Data Loading & Exploration

In [ ]:
# Try Kaggle path first, then HuggingFace
import glob

try:
    kaggle_paths = glob.glob("/kaggle/input/*/train.csv")
    if kaggle_paths:
        DATA_DIR = os.path.dirname(kaggle_paths[0])
        train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
        test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
        print(f"Loaded from Kaggle: {DATA_DIR}")
    else:
        raise FileNotFoundError
except Exception:
    from datasets import load_dataset
    DATASET_REVISION = "3afc18f0b582b3cfdb927822cff57ddc6e871f9c"
    ds = load_dataset("ailsntua/QEvasion", revision=DATASET_REVISION)
    train_df = ds["train"].to_pandas()
    test_df  = ds["test"].to_pandas()
    print("Loaded from HuggingFace: ailsntua/QEvasion")

print(f"\nTrain samples : {len(train_df)}")
print(f"Test  samples : {len(test_df)}")
print(f"Columns       : {list(train_df.columns)}")
train_df.head(3)

In [ ]:
print("=" * 55)
print("CLASS DISTRIBUTION (train)")
print("=" * 55)
class_counts = train_df["clarity_label"].value_counts()
print(class_counts)
print(f"\nProportions:\n{(class_counts / len(train_df)).round(3)}")

# Missing values
print(f"\nMissing values:")
for col in ["question", "interview_answer", "clarity_label"]:
    print(f"  {col}: {train_df[col].isna().sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
labels_order = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
counts = [class_counts.get(label, 0) for label in labels_order]

axes[0].bar(labels_order, counts, color=colors)
axes[0].set_title("Training class counts")
axes[0].set_ylabel("Examples")
axes[0].tick_params(axis="x", rotation=15)

axes[1].pie(counts, labels=labels_order, colors=colors, autopct="%1.1f%%")
axes[1].set_title("Training class proportions")

plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
train_df["question_len"] = train_df["question"].fillna("").str.split().str.len()
train_df["answer_len"]   = train_df["interview_answer"].fillna("").str.split().str.len()
train_df["combined_len"] = train_df["question_len"] + train_df["answer_len"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_cls = {"Clear Reply": "#2ecc71", "Ambivalent": "#f39c12", "Clear Non-Reply": "#e74c3c"}

for i, (col, title) in enumerate([
    ("question_len", "Question Length (words)"),
    ("answer_len", "Answer Length (words)"),
    ("combined_len", "Combined Length (words)"),
]):
    for lbl, clr in colors_cls.items():
        subset = train_df[train_df["clarity_label"] == lbl]
        axes[i].hist(subset[col], bins=30, alpha=0.5, label=lbl, color=clr)
    axes[i].set_title(title, fontsize=12, fontweight="bold")
    axes[i].set_xlabel("Word Count")
    axes[i].set_ylabel("Frequency")
    axes[i].legend()

plt.tight_layout()
plt.savefig("text_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("Length statistics by class:")
for lbl in labels_order:
    sub = train_df[train_df["clarity_label"] == lbl]
    print(f"\n  {lbl}:")
    print(f"    Question — mean {sub['question_len'].mean():.1f}  median {sub['question_len'].median():.0f}")
    print(f"    Answer   — mean {sub['answer_len'].mean():.1f}  median {sub['answer_len'].median():.0f}")

## 2 · Data Preprocessing

### Input Construction

We feed each example to the transformer as a **sentence pair**:

```
[CLS] question [SEP] answer [SEP]
```

This leverages the model's pre-trained next-sentence / sentence-pair understanding
and lets the self-attention jointly reason over question and answer tokens.

In [ ]:
# Encode labels
train_df["label"] = train_df["clarity_label"].map(LABEL2ID)

# Fill missing text
for col in ["question", "interview_answer"]:
    train_df[col] = train_df[col].fillna("")
    test_df[col]  = test_df[col].fillna("")

# Stratified train / validation split
X_train_q, X_val_q, X_train_a, X_val_a, y_train, y_val = train_test_split(
    train_df["question"].values,
    train_df["interview_answer"].values,
    train_df["label"].values,
    test_size=0.15,
    random_state=SEED,
    stratify=train_df["label"].values,
)

print(f"Training   : {len(y_train)}")
print(f"Validation : {len(y_val)}")
print(f"\nTrain label distribution:")
for name, lid in LABEL2ID.items():
    n = (y_train == lid).sum()
    print(f"  {name}: {n}  ({n / len(y_train) * 100:.1f}%)")

In [ ]:
class_weights = compute_class_weight(
    "balanced", classes=np.unique(y_train), y=y_train
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class weights (balanced): {class_weights.round(4)}")

## 3 · Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer : {type(tokenizer).__name__}")
print(f"Vocab size: {tokenizer.vocab_size}")

# Quick demo
demo_enc = tokenizer("Is the economy improving?",
                     "Well, there are many factors at play here.",
                     max_length=64, truncation=True, padding="max_length")
print(f"\nDemo tokens (first 25):")
print(tokenizer.convert_ids_to_tokens(demo_enc["input_ids"][:25]))

In [ ]:
token_lengths = []
for q, a in zip(train_df["question"].values, train_df["interview_answer"].values):
    enc = tokenizer(str(q), str(a), truncation=False)
    token_lengths.append(len(enc["input_ids"]))
token_lengths = np.array(token_lengths)

print(f"Token-length statistics (question + answer):")
print(f"  Mean   : {token_lengths.mean():.1f}")
print(f"  Median : {np.median(token_lengths):.1f}")
print(f"  P95    : {np.percentile(token_lengths, 95):.0f}")
print(f"  P99    : {np.percentile(token_lengths, 99):.0f}")
print(f"  Max    : {token_lengths.max()}")
print(f"  ≤ {MAX_LEN} tokens: {(token_lengths <= MAX_LEN).mean() * 100:.1f}%")

plt.figure(figsize=(10, 4))
plt.hist(token_lengths, bins=50, edgecolor="black", alpha=0.7, color="#3498db")
plt.axvline(MAX_LEN, color="red", linestyle="--", linewidth=2, label=f"MAX_LEN = {MAX_LEN}")
plt.xlabel("Token Length")
plt.ylabel("Frequency")
plt.title("Token-Length Distribution (Question + Answer)", fontsize=13, fontweight="bold")
plt.legend()
plt.tight_layout()
plt.savefig("token_length_dist.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
class ClarityDataset(Dataset):
    '''PyTorch Dataset for clarity classification.'''

    def __init__(self, questions, answers, labels, tokenizer, max_len):
        self.questions = questions
        self.answers   = answers
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.questions[idx]),
            str(self.answers[idx]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

train_dataset = ClarityDataset(X_train_q, X_train_a, y_train, tokenizer, MAX_LEN)
val_dataset   = ClarityDataset(X_val_q,   X_val_a,   y_val,   tokenizer, MAX_LEN)
test_dataset  = ClarityDataset(
    test_df["question"].values,
    test_df["interview_answer"].values,
    None, tokenizer, MAX_LEN,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True,
                          worker_init_fn=seed_worker, generator=g)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

## 4 · Model Development

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model            : {MODEL_NAME}")
print(f"Architecture     : {type(model).__name__}")
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")

## 5 · Training & Optimization

**Optimizer**: AdamW with differential weight decay (no decay on bias / LayerNorm).
**Scheduler**: Linear warm-up (10 % of steps) followed by linear decay.
**Loss**: Weighted cross-entropy to handle class imbalance.
**Gradient clipping**: max-norm = 1.0.
**Early stopping**: patience = 2 epochs on validation weighted F1.

In [ ]:
no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias", "layernorm.weight", "layernorm.bias"]
optimizer_grouped = [
    {"params": [p for n, p in model.named_parameters()
                if not any(nd in n for nd in no_decay)],
     "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters()
                if any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
]

optimizer = torch.optim.AdamW(optimizer_grouped, lr=LEARNING_RATE, eps=1e-8)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

print(f"Total steps  : {total_steps}")
print(f"Warmup steps : {warmup_steps}")
print(f"LR           : {LEARNING_RATE}")

In [ ]:
def model_forward(model, batch, device):
    '''Build model inputs, handling token_type_ids compatibility.'''
    inputs = {
        "input_ids":      batch["input_ids"].to(device),
        "attention_mask":  batch["attention_mask"].to(device),
    }
    if USE_TOKEN_TYPE_IDS and "token_type_ids" in batch:
        inputs["token_type_ids"] = batch["token_type_ids"].to(device)
    return model(**inputs)

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for batch in loader:
        optimizer.zero_grad()
        outputs = model_forward(model, batch, device)
        loss = criterion(outputs.logits, batch["labels"].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()
        all_preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())
    n = len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    w_f1     = f1_score(all_labels, all_preds, average="weighted")
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / n, acc, w_f1, macro_f1


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    for batch in loader:
        outputs = model_forward(model, batch, device)
        loss = criterion(outputs.logits, batch["labels"].to(device))
        running_loss += loss.item()
        probs = torch.softmax(outputs.logits, dim=1)
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())
    n = len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    w_f1     = f1_score(all_labels, all_preds, average="weighted")
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / n, acc, w_f1, macro_f1, np.array(all_preds), np.array(all_labels), np.array(all_probs)

In [ ]:
history = {k: [] for k in [
    "train_loss", "train_acc", "train_f1", "train_macro_f1",
    "val_loss",   "val_acc",   "val_f1",   "val_macro_f1",
]}
best_val_f1, best_epoch, patience_ctr = 0, 1, 0

print("=" * 70)
print(f"  Training {MODEL_NAME}")
print("=" * 70)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc, tr_f1, tr_mf1 = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion, device
    )
    vl_loss, vl_acc, vl_f1, vl_mf1, vl_preds, vl_labels, vl_probs = evaluate(
        model, val_loader, criterion, device
    )
    elapsed = time.time() - t0

    for key, val in [
        ("train_loss", tr_loss), ("train_acc", tr_acc),
        ("train_f1", tr_f1),     ("train_macro_f1", tr_mf1),
        ("val_loss", vl_loss),   ("val_acc", vl_acc),
        ("val_f1", vl_f1),       ("val_macro_f1", vl_mf1),
    ]:
        history[key].append(val)

    print(f"\nEpoch {epoch}/{EPOCHS}  ({elapsed:.0f}s)")
    print(f"  Train — loss {tr_loss:.4f}  acc {tr_acc:.4f}  F1 {tr_f1:.4f}  macroF1 {tr_mf1:.4f}")
    print(f"  Val   — loss {vl_loss:.4f}  acc {vl_acc:.4f}  F1 {vl_f1:.4f}  macroF1 {vl_mf1:.4f}")

    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        best_epoch = epoch
        patience_ctr = 0
        torch.save(model.state_dict(), "best_model.pt")
        print(f"  ✓ New best model saved (F1={vl_f1:.4f})")
    else:
        patience_ctr += 1
        print(f"  – No improvement ({patience_ctr}/{PATIENCE})")

    if patience_ctr >= PATIENCE:
        print(f"\n⚠ Early stopping at epoch {epoch}")
        break

print(f"\n{'=' * 70}")
print(f"Best epoch {best_epoch}  —  Val weighted-F1 = {best_val_f1:.4f}")
print(f"{'=' * 70}")

In [ ]:
model.load_state_dict(torch.load("best_model.pt", map_location=device, weights_only=True))
print(f"Loaded best checkpoint (epoch {best_epoch})")

## 6 · Evaluation & Results

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

plots = [
    ("train_loss", "val_loss", "Loss"),
    ("train_acc",  "val_acc",  "Accuracy"),
    ("train_f1",   "val_f1",   "Weighted F1"),
    ("train_macro_f1", "val_macro_f1", "Macro F1"),
]
for ax, (tr_key, vl_key, title) in zip(axes.flat, plots):
    ax.plot(epochs_range, history[tr_key], "b-o", label="Train")
    ax.plot(epochs_range, history[vl_key], "r-o", label="Validation")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f"{MODEL_NAME} — Training History", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
_, val_acc, val_f1, val_mf1, val_preds, val_labels, val_probs = evaluate(
    model, val_loader, criterion, device
)

print("=" * 60)
print(f"FINAL RESULTS — {MODEL_NAME}")
print("=" * 60)
print(f"\n  Accuracy       : {val_acc:.4f}")
print(f"  Weighted F1    : {val_f1:.4f}")
print(f"  Macro F1       : {val_mf1:.4f}")
print(f"\n{classification_report(val_labels, val_preds, target_names=list(LABEL2ID.keys()), digits=4)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cm = confusion_matrix(val_labels, val_preds)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
class_names = list(LABEL2ID.keys())

for ax, data, fmt, title in [
    (axes[0], cm, "d", "Confusion Matrix (counts)"),
    (axes[1], cm_norm, ".3f", "Confusion Matrix (normalized)"),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("True", fontsize=12)
    ax.set_title(f"{MODEL_NAME}\n{title}", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
prec, rec, f1c, sup = precision_recall_fscore_support(
    val_labels, val_preds, average=None
)
metrics_df = pd.DataFrame({
    "Class": class_names,
    "Precision": prec,
    "Recall": rec,
    "F1-Score": f1c,
    "Support": sup.astype(int),
})
print(metrics_df.to_string(index=False))

x = np.arange(NUM_CLASSES)
w = 0.22
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, prec, w, label="Precision", color="#3498db")
ax.bar(x,     rec,  w, label="Recall",    color="#2ecc71")
ax.bar(x + w, f1c,  w, label="F1-Score",  color="#e74c3c")
ax.set_xticks(x)
ax.set_xticklabels(class_names)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title(f"{MODEL_NAME} — Per-Class Metrics", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("per_class_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 · Error Analysis

In [ ]:
val_df = pd.DataFrame({
    "question":   X_val_q,
    "answer":     X_val_a,
    "true_label": [ID2LABEL[l] for l in val_labels],
    "pred_label": [ID2LABEL[p] for p in val_preds],
    "correct":    val_labels == val_preds,
    "confidence": val_probs.max(axis=1),
})
val_df["question_len"] = val_df["question"].str.split().str.len()
val_df["answer_len"]   = val_df["answer"].str.split().str.len()

errors = val_df[~val_df["correct"]]
print(f"Total errors : {len(errors)} / {len(val_df)}  ({len(errors)/len(val_df)*100:.1f}%)")
print(f"\nErrors by TRUE label:")
for lbl in class_names:
    n_err = len(errors[errors["true_label"] == lbl])
    n_tot = len(val_df[val_df["true_label"] == lbl])
    print(f"  {lbl:20s}: {n_err:3d} / {n_tot:3d}  ({n_err/max(1,n_tot)*100:5.1f}%)")

print(f"\nMost common misclassification patterns:")
patterns = errors.groupby(["true_label", "pred_label"]).size().sort_values(ascending=False)
for (tl, pl), cnt in patterns.head(6).items():
    print(f"  {tl:20s} → {pl:20s} : {cnt}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(val_df[val_df["correct"]]["confidence"],  bins=30, alpha=0.6,
             label="Correct", color="#2ecc71")
axes[0].hist(val_df[~val_df["correct"]]["confidence"], bins=30, alpha=0.6,
             label="Incorrect", color="#e74c3c")
axes[0].set_title("Confidence Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Max Predicted Probability")
axes[0].set_ylabel("Count")
axes[0].legend()

# Per-class confidence for errors
for lbl, clr in colors_cls.items():
    sub = errors[errors["true_label"] == lbl]
    if len(sub) > 0:
        axes[1].hist(sub["confidence"], bins=15, alpha=0.5, label=lbl, color=clr)
axes[1].set_title("Error Confidence by True Class", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Max Predicted Probability")
axes[1].legend()

plt.tight_layout()
plt.savefig("confidence_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Mean confidence — correct:   {val_df[val_df['correct']]['confidence'].mean():.4f}")
print(f"Mean confidence — incorrect: {val_df[~val_df['correct']]['confidence'].mean():.4f}")

In [ ]:
val_df["answer_len_cat"] = pd.cut(
    val_df["answer_len"],
    bins=[0, 50, 100, 200, 9999],
    labels=["Short (≤50)", "Medium (51-100)", "Long (101-200)", "Very Long (>200)"],
)
val_df["question_len_cat"] = pd.cut(
    val_df["question_len"],
    bins=[0, 10, 20, 40, 9999],
    labels=["Short (≤10)", "Medium (11-20)", "Long (21-40)", "Very Long (>40)"],
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, cat_col, title, clr in [
    (axes[0], "answer_len_cat",   "Accuracy by Answer Length",   "#3498db"),
    (axes[1], "question_len_cat", "Accuracy by Question Length", "#e74c3c"),
]:
    grp = val_df.groupby(cat_col, observed=False)["correct"].agg(["mean", "count"])
    bars = ax.bar(range(len(grp)), grp["mean"], color=clr, edgecolor="black", linewidth=0.5)
    ax.set_xticks(range(len(grp)))
    ax.set_xticklabels(grp.index, rotation=15)
    ax.set_ylim(0, 1.05)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("Accuracy")
    ax.grid(True, alpha=0.3, axis="y")
    for i, (_, row) in enumerate(grp.iterrows()):
        ax.text(i, row["mean"] + 0.02, f"n={int(row['count'])}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("subgroup_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print("Detailed subgroup accuracy:")
for cat_col in ["answer_len_cat", "question_len_cat"]:
    print(f"\n{cat_col}:")
    grp = val_df.groupby(cat_col, observed=False)["correct"].agg(["mean", "count"])
    for idx, row in grp.iterrows():
        print(f"  {str(idx):25s}  acc={row['mean']:.4f}  n={int(row['count'])}")

In [ ]:
# If the dataset contains evasion_label, analyze performance by evasion type
if "evasion_label" in train_df.columns:
    # Rebuild val_df with evasion labels from the original split
    _, val_indices = train_test_split(
        range(len(train_df)), test_size=0.15, random_state=SEED,
        stratify=train_df["label"].values,
    )
    val_evasion = train_df.iloc[val_indices]["evasion_label"].values
    val_df = val_df.copy()
    val_df["evasion_label"] = val_evasion

    print("Accuracy by evasion label:")
    for ev_label in sorted(val_df["evasion_label"].dropna().unique()):
        sub = val_df[val_df["evasion_label"] == ev_label]
        if len(sub) >= 5:
            print(f"  {ev_label:30s}  acc={sub['correct'].mean():.4f}  n={len(sub)}")

## 8 · Prediction Export

In [ ]:
model.eval()
test_preds, test_probs = [], []

with torch.no_grad():
    for batch in test_loader:
        outputs = model_forward(model, batch, device)
        probs = torch.softmax(outputs.logits, dim=1)
        test_preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())
        test_probs.extend(probs.cpu().numpy())

test_pred_labels = [ID2LABEL[p] for p in test_preds]

# If the test set has labels (HuggingFace version), evaluate
if "clarity_label" in test_df.columns and test_df["clarity_label"].notna().all():
    test_true = test_df["clarity_label"].map(LABEL2ID).values
    test_acc  = accuracy_score(test_true, test_preds)
    test_f1   = f1_score(test_true, test_preds, average="weighted")
    test_mf1  = f1_score(test_true, test_preds, average="macro")
    print(f"Test Accuracy    : {test_acc:.4f}")
    print(f"Test Weighted F1 : {test_f1:.4f}")
    print(f"Test Macro F1    : {test_mf1:.4f}")
    print(f"\n{classification_report(test_true, test_preds, target_names=class_names, digits=4)}")
else:
    print("Test set has no labels — skipping evaluation.")

In [ ]:
def export_predictions(test_ids, predicted_labels, filename="submission.csv"):
    """
    Export model predictions to a CSV with columns 'Id' and 'Predicted'.

    Args:
        test_ids:          List or array of test set indices.
        predicted_labels:  List or array of predicted class labels.
        filename:          Output CSV filename.
    """
    submission = pd.DataFrame({
        "Id":        test_ids,
        "Predicted": predicted_labels,
    })
    submission.to_csv(filename, index=False)
    print(f"Saved: {filename}  ({len(submission)} rows)")
    print(submission["Predicted"].value_counts().to_string())
    return submission


In [ ]:
# Resolve test IDs
if "index" in test_df.columns:
    sub_ids = test_df["index"].values
elif "id" in test_df.columns:
    sub_ids = test_df["id"].values
else:
    sub_ids = np.arange(len(test_df))

submission = export_predictions(sub_ids, test_pred_labels, SUBMISSION_FILE)
submission.head(10)


## Results pending rerun

After a clean end-to-end run, record the best epoch, validation accuracy, weighted F1, and macro F1 here together with the environment and dataset revision.